In [1]:
from DWT import DWT_2D
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
from PIL import Image
import torch
import json

In [2]:
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().squeeze().numpy()
    else:
        x = np.squeeze(x)
    return x

In [3]:
def show_wavelet_decomposed_Xray_with_bbox(img_path, bbox):
    img = Image.open(img_path)
    img = np.array(img)[:,:,0]
    img = torch.tensor(img, dtype=torch.float32)
    img = img.reshape(1,1,512,512)
    img_ll, img_lh, img_hl, img_hh = dwt2D(img)
    img_np = to_numpy(img)
    img_ll_np = to_numpy(img_ll)
    img_lh_np = to_numpy(img_lh)
    img_hl_np = to_numpy(img_hl)
    img_hh_np = to_numpy(img_hh)
    img_lh_hl_np = img_lh_np + img_hl_np
    imgs_list = [img_np, img_ll_np, img_lh_np, img_hl_np, img_hh_np, img_lh_hl_np]
    title_list = ['original X-ray', 'shape / approximation', 'horizontal detail', 'vertical detail', 'noise / diagonal detail', 'appearance / horizontal+vertical detail']
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()
    
    for i, ax in enumerate(axes):
        current_img = imgs_list[i].squeeze() 

        ax.imshow(current_img, cmap='gray')
        ax.set_title(title_list[i])
        ax.axis('off')

        scale_factor = 1.0 if i == 0 else 0.5

        x = bbox[0] * scale_factor
        y = bbox[1] * scale_factor
        w = bbox[2] * scale_factor
        h = bbox[3] * scale_factor
    
        rect = patches.Rectangle(
            (x, y), 
            w, 
            h, 
            linewidth=2, 
            edgecolor='r', 
            facecolor='none'
        )
        
        ax.add_patch(rect)
    
    plt.tight_layout()
    plt.show()

In [4]:
wavename = 'haar'
dwt2D = DWT_2D(wavename)

In [ ]:
anno_path = ''
with open(anno_path, 'r') as f:
    annos = json.load(f)

In [ ]:
img_path = ""
bbox = annos["annotations"][0]['bbox']
show_wavelet_decomposed_Xray_with_bbox(img_path, bbox)